# Day 19 — GPU Acceleration & CUDA

## 1. Learning Objectives
- Understand why GPUs are used in Deep Learning.
- Learn how to check for GPU availability (`torch.cuda.is_available()`).
- Define a device-agnostic setup.
- Move Tensors and Models to the GPU using `.to(device)`.
- Diagnose and fix Device Mismatch errors.

## 2. Prerequisites
- Tensors (Day 2)
- Training Loops (Day 12)

In [ ]:
import torch
import torch.nn as nn

## 3. Concept Explanation
CPUs are like a few highly intelligent professors. They can solve very complex sequential tasks. GPUs are like thousands of high school students. They aren't as smart individually, but they can perform thousands of simple math problems (like Matrix Multiplication) at the exact same time.

Deep Learning is almost entirely Matrix Multiplication. Therefore, running a Neural Network on a GPU can be 10x to 100x faster than running it on a CPU.

## 4. Device Agnostic Code
You want your code to run on a GPU if one is available, but fallback to a CPU if it's not (e.g., when sharing code with a friend who doesn't have a gaming PC). We do this using `torch.device`.

In [ ]:
# 8. Simple Example: Device Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 9. Moving Data and Models
By default, all tensors and models are created on the CPU memory. To utilize the GPU, you must explicitly copy them over using `.to(device)`.

In [ ]:
# 1. Moving a Tensor
x = torch.randn(5, 5) # Created on CPU
x = x.to(device)      # Copied to GPU (if available)
print("Tensor device:", x.device)

# 2. Moving a Model
model = nn.Linear(5, 2)
model = model.to(device)

# Check where the model's weights live
print("Model weights device:", next(model.parameters()).device)

## 10. Experiment: The Golden Rule of Devices
**PyTorch Golden Rule**: Tensors cannot interact if they are on different devices. You cannot multiply a CPU tensor by a GPU tensor.

In [ ]:
cpu_tensor = torch.ones(2, 2) # Stays on CPU
gpu_tensor = torch.ones(2, 2).to(device) # Moves to GPU

# Only runs if you actually have a GPU, otherwise they are both on CPU
if torch.cuda.is_available():
    try:
        result = cpu_tensor + gpu_tensor
    except RuntimeError as e:
        print(f"Error Caught!\n{e}")

## 11. Practice Exercise 1: Update the Training Loop
Look at this pseudo-code training loop. Where exactly do you need to add `.to(device)` to make it run on a GPU?

In [ ]:
# model = MyModel()
# for images, labels in dataloader:
#     preds = model(images)
#     loss = criterion(preds, labels)
#     ... backward and step ...

In [ ]:
# SOLUTION
# 1. Model must go to device BEFORE the loop (or before setting up the optimizer)
# model = MyModel().to(device)
# 
# for images, labels in dataloader:
#     # 2. Data must go to device AS SOON AS it comes out of the dataloader
#     images = images.to(device)
#     labels = labels.to(device)
#     
#     preds = model(images)
#     loss = criterion(preds, labels)
#     ... backward and step ...

## 13. Debugging Challenge
You trained a model on the GPU. You want to plot the predictions using `matplotlib` (which runs on the CPU and requires NumPy arrays).
Why does `plt.plot(predictions.numpy())` crash, and how do you fix it?

**Solution:** NumPy cannot read memory that lives on the GPU. You must first copy the tensor back to the CPU memory using `.cpu()`, and since it's likely attached to the computational graph, you also need to detach it. 

The fix: `plt.plot(predictions.detach().cpu().numpy())`

## 17. Interview Questions
1. **What happens if you initialize your Optimizer BEFORE moving your model to the GPU?**
   *Answer*: The optimizer will track the CPU parameters. When you move the model to the GPU, PyTorch creates a *copy* of the parameters on the GPU. The optimizer will update the old CPU parameters, and your GPU model will never learn. ALWAYS do `model = model.to(device)` BEFORE `optim.Adam(model.parameters())`.
2. **What does `torch.cuda.empty_cache()` do?**
   *Answer*: It releases all unoccupied cached memory currently held by the caching allocator so that it can be used in other GPU applications. However, it does NOT free memory occupied by active tensors. You usually don't need to call this manually.

## 19. Day Summary
- Write device-agnostic code using `torch.device('cuda' if torch.cuda.is_available() else 'cpu')`.
- Send models to the device `model.to(device)` *before* initializing the optimizer.
- Send data batches to the device `x.to(device)` immediately inside the training loop.
- NumPy requires data to be on the CPU: `.detach().cpu().numpy()`.